# First Real Pressure Curve

Phase 1 milestone: the transducer, bench-wired to a Pico 2 W and screwed into the La Pavoni's piston shaft adapter, captured a real espresso pull for the first time.

Data: `data/first_shot_2026-09-06.csv` (archived copy of the Pico's `readings.csv` from this run, columns `t` (seconds since script start), `raw` (ADC 0-65535), `voltage`, `bar`). Calibration: `bar = voltage / 3.3 * 16`, assuming 0V = 0 bar and the sensor's rated max 3.3V = 16 bar.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/first_shot_2026-09-06.csv")

# Trim the long flat baseline before the pull actually started
df = df[df["t"] >= 80].reset_index(drop=True)
df.head()

In [ ]:
peak_row = df.loc[df["bar"].idxmax()]
baseline = df[df["t"] < 105]["bar"].mean()  # flat region just before the rise starts

TARGET_BAR = 9      # what we're actually trying to hit
THRESHOLD_BAR = 6   # "meaningful extraction" cutoff

above = df["bar"] >= THRESHOLD_BAR
extraction_start = df.loc[above, "t"].min()
extraction_end = df.loc[above, "t"].max()

dt = df["t"].diff()
time_above_threshold = dt[above].sum()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["t"], df["bar"], linewidth=1.5)

# Baseline reference
ax.axhline(baseline, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.text(df["t"].iloc[0], baseline, f"baseline ~{baseline:.2f} bar ",
        va="bottom", ha="left", color="gray", fontsize=9)

# Target pressure reference
ax.axhline(TARGET_BAR, color="seagreen", linestyle=":", linewidth=1.2, alpha=0.8)
ax.text(df["t"].iloc[0], TARGET_BAR, f"target {TARGET_BAR:.0f} bar ",
        va="bottom", ha="left", color="seagreen", fontsize=9)

# Meaningful-extraction threshold + shaded window
ax.axhline(THRESHOLD_BAR, color="darkorange", linestyle=":", linewidth=1.2, alpha=0.8)
ax.axvspan(extraction_start, extraction_end, color="darkorange", alpha=0.12)
ax.text((extraction_start + extraction_end) / 2, THRESHOLD_BAR - 0.6,
        f"meaningful extraction\n~{time_above_threshold:.1f}s",
        va="top", ha="center", color="darkorange", fontsize=9)

# Peak annotation
ax.plot(peak_row["t"], peak_row["bar"], "o", color="crimson")
ax.annotate(
    f"peak: {peak_row['bar']:.2f} bar\n(t={peak_row['t']:.1f}s)",
    xy=(peak_row["t"], peak_row["bar"]),
    xytext=(peak_row["t"] + 10, peak_row["bar"]),
    va="center",
    fontsize=9,
    color="crimson",
    arrowprops=dict(arrowstyle="->", color="crimson"),
)

ax.set_xlabel("Time (s)")
ax.set_ylabel("Pressure (bar)")
ax.set_title("La Pavoni pull — pressure over time")
ax.set_ylim(0, max(TARGET_BAR, df["bar"].max()) + 1)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Quick stats on the pull (peak_row/baseline computed in the plot cell above)
print(f"Baseline (pre-pull) pressure: {baseline:.2f} bar")
print(f"Peak pressure: {peak_row['bar']:.2f} bar at t={peak_row['t']:.2f}s")

In [ ]:
# Did we reach target, and how long did we spend in the "meaningfully extracting" zone?
shortfall = TARGET_BAR - peak_row["bar"]
status = "reached target" if shortfall <= 0 else f"fell short by {shortfall:.2f} bar"

print(f"Target pressure: {TARGET_BAR} bar")
print(f"Peak reached: {peak_row['bar']:.2f} bar ({status})")
print(f"Meaningful extraction time (>= {THRESHOLD_BAR} bar): {time_above_threshold:.2f}s")

## Takeaway

This first pull never reached the 9 bar target (peaked at ~7.4 bar) and only spent ~2.7 seconds above the 6 bar "meaningfully extracting" threshold before decaying away. That lines up with how the shot actually tasted β€” sour, i.e. underextracted β€” and with the puck resistance not lasting long during the pull. First real data point tying a felt outcome (sour shot) to a measured cause (insufficient pressure, held for too short a time), rather than just a guess.